In [1]:
# chromid reference data
import pandas as pd
import textract

file_path = "/active-data/analysis_results/chr_pla/reference_data/PMID20080407/1-s2.0-S0966842X09002698-mmc1.doc"
content = textract.process(file_path, encoding="utf-8")
content = content.decode("utf-8")

ref1_list = []
for line in content.splitlines():
    if "Chromid" in line and '|' in line:
        parts = line.split("|")
        nc_code = parts[4].strip()
        ref1_list.append(nc_code.replace(' ', '_'))

file_path = "/active-data/analysis_results/chr_pla/reference_data/PMID40827884/msystems.00175-25-s0003.xlsx"

df = pd.read_excel(file_path, skiprows=1)
chromid_ref = df[df['GC_difference_%'] != 0]
ref2_list = chromid_ref['Replicon_Accession'].to_list()

In [2]:
# chromid-finder data
import ast
import os

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
filted_data = all_data[all_data['genus_clean'].isin(keep_genus)]

found_chromids = []
chromid_dir = '/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/chromid_results'
for acc_n in filted_data['accession']:
    file_path = f'{chromid_dir}/{acc_n}.txt'
    
    if os.path.getsize(file_path) == 0:
        continue
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
        for i, line in enumerate(lines):
            if "Possible bacterial chromids:" in line:
                if i + 1 >= len(lines):
                    continue
                chromid_line = lines[i+1].strip()
                if not chromid_line or chromid_line == '------':
                    break
                
                chromid_list = [c.strip() for c in chromid_line.split(",") if c.strip()]
                for chromid in chromid_list:
                    found_chromids.append(chromid.split('.')[0])

In [3]:
# all replicon
all_result = []
for genus_name in keep_genus:
    os.chdir(f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}')
    contig_info = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    all_result.append(contig_info)
all_result = pd.concat(all_result, ignore_index=True)
all_result['contig'] = all_result['accession'].apply(
    lambda x: x.split('-')[-1].split('.')[0]
)

In [4]:
all_chromids = set(ref1_list + ref2_list + found_chromids)
chromid_result = all_result[all_result['contig'].isin(all_chromids)]
print(len(ref1_list + ref2_list + found_chromids), len(all_chromids), len(chromid_result))

1929 1410 742


In [5]:
print(chromid_result['category-pident_90'].value_counts())
print(all_result['category-pident_90'].value_counts())

category-pident_90
typical chromosome       717
typical plasmid           24
transitional replicon      1
Name: count, dtype: int64
category-pident_90
typical plasmid          41285
typical chromosome       30011
transitional replicon      968
Name: count, dtype: int64
